# **Static Obstacle Avoidance** 🚧🏎️

---

**What this notebook does:** Now that we have successfully deployed a racing policy on our car, lets train a policy that reacts to obstacles with
using our lidar in real time, instead of memorizing one safe line around one fixed obstacle for a head-to-head racing scenerio.

**How:** every episode, we bake 1-3 circular obstacles — sized close to our
own car — into the track's occupancy grid, at a new random position and size. Because the obstacle becomes part of the same grid the lidar
already scans against, nothing about our lidar or collision code has to
change — the car just sees "wall" exactly where we painted it. Since the
obstacle is never in the same place twice, the only way the policy can survive
is to genuinely read the lidar and react.

📌 In this notebook we shall use the same from-scratch PPO algorithm as Learn to Drive — only the environment
and rewards change.

---


## 📋 How This Works

We build a pool of maps, each with a few obstacles baked directly into the
track image, and hand the car a different one every episode:

1. **Load the track's occupancy grid** (the same grid the lidar already scans against).
2. **Paint 1-3 random circles onto a copy of it** — random position along
   the centerline, random radius close to our car's own size.
3. **Save many such variants** ahead of time (a "pool"), instead of just one.
4. **Pick a fresh, unseen variant every `reset()`** during training.

Because the obstacle lives in the same grid as the walls, our existing
lidar/collision pipeline needs **zero changes** — the gym just treats a
painted obstacle like a wall.


<p align="left">
  <img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/static_obs.gif" width="530"/>
</p>


---


---

## 📝 Student TODOs — Complete These Before Running Training

This notebook has **6 TODOs** to complete. TODOs 5–6 are the PPO pieces from
Learn to Drive — paste in your own working versions if you have them. Everything else is given to you.

| # | TODO | Location | What You Build |
|---|---|---|---|
| **TODO 1** | World <-> pixel coordinate conversion | Section 2 — `_world_to_pixel` | Coordinate transform |
| **TODO 2** | Sample a random centerline point | Section 2 — `_sample_centerline_point` | Obstacle placement |
| **TODO 3** | Paint a circular obstacle onto the grid | Section 2 — `generate_obstacle_map` | Obstacle generation |
| **TODO 4** | Design the obstacle-avoidance reward | Section 3 — Wrapper `step()` | Reward shaping |
| **TODO 5** | Actor-critic forward pass | Section 5 — `ActorCritic.forward` | Same as Learn to Drive |
| **TODO 6** | Compute GAE advantages | Section 5.2 — `compute_gae` | Same as Learn to Drive |

> **📌 Tip:** Read each section's explanation carefully before attempting its TODO — the
> math/logic you need is explained right above the corresponding code cell.

---


---

# **Section 1 — Setup** 🔧

In [ ]:
# ── Install all dependencies ──────────────────────────────────────────────────
# f1tenth_gym: the race car simulator (dev-humble branch = latest stable)
!pip install -q git+https://github.com/f1tenth/f1tenth_gym.git@dev-humble

# stable-baselines3: we use only its VecEnv utilities (parallel envs, reward
# normalization) for running rollouts — the PPO algorithm itself is ours
!pip install -q stable-baselines3[extra]

# wandb: experiment tracking — we will log metrics here in real time
!pip install -q wandb

print("Installation complete!")

## 1.2 Verify Imports 📦

In [ ]:
# ── Verify everything imports correctly ───────────────────────────────────────

import os
import yaml
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# This gym reuses observation buffers for speed, which gymnasium's passive
# checker flags on every reset/step — harmless here, so we quiet it (plus a
# jupyter_client deprecation notice neither we nor the gym control).
warnings.filterwarnings("ignore", message=".*share an object.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

import gymnasium as gym
import torch
import stable_baselines3
import wandb
import f1tenth_gym
from f1tenth_gym.envs.track import Track

print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
print(f"Stable Baselines: {stable_baselines3.__version__}")
print(f"Gymnasium:        {gym.__version__}")
print()
print("All good — let's start training!")


## 1.3 Pre-Cache the Track Map 🗺️

In [ ]:
# ── Pre-cache the track map ───────────────────────────────────────────────────
Track.from_track_name('Spielberg')
print("Spielberg track cached — ready to train!")

---

# **Section 2 — Generating Randomized Obstacle Maps** 🧱

## 2.1 World Coordinates ↔ Pixel Coordinates 📐

The track map is an image: black pixels are walls/obstacles, white pixels are
free space. A YAML file next to it tells us the `resolution` (metres per
pixel) and the `origin` (where the image's bottom-left corner sits in the
world). To paint an obstacle at a real-world point, we first need to turn
that world point into a pixel.

One consideration: in a standard map image, row 0 is the **top**, while world-$y$ increases
**upward**. But this gym's `occupancy_map` array is already stored with row 0 at the
low-$y$ edge (the overlay sanity check below confirms it), so **no row flip** is needed:

$$\text{col} = \frac{x - x_{\text{origin}}}{\text{resolution}} \qquad
\text{row} = \frac{y - y_{\text{origin}}}{\text{resolution}}$$


## 2.2 The Generation Strategy 🧩

For every variant we want to create, we repeat four simple steps:

1. **Pick a spot** — a random point along the track centerline (so the
   obstacle sits somewhere the car would otherwise drive), skipping the area
   right at the spawn point so the car isn't dropped on top of an obstacle.
2. **Pick a size and count** — 1 to 3 obstacles, each a circle with a random
   radius of 0.2–0.5 m, close to our own car's size (car is 0.31m × 0.58m).
3. **Paint it** — convert the obstacle's world position to pixels and draw a
   filled circle into a **copy** of the base map.
4. **Save it** — write the new image + a matching YAML under a new map name.

We repeat this many times to build a **pool of variants**, so training can
hand out a fresh, never-before-seen layout at every episode.

<p align="left">
  <img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/static_obs.png" width="430"/>
</p>


## 2.3 Generator Functions ⚙️

Loads the base map through the gym itself and reads the real occupancy grid straight off the resulting
`Track` object — no guessing at file paths.


In [ ]:
# ── Obstacle Map Generator ──────────────────────────────────────────────────────
# Reads the base map's occupancy grid straight from the gym's Track object,
# paints obstacles onto a copy, and saves each variant as a new map file.



BASE_MAP_NAME     = "Spielberg"     # track to add obstacles to
NUM_VARIANTS      = 40              # size of the randomized obstacle-map pool
MIN_OBSTACLES     = 1
MAX_OBSTACLES     = 3
OBSTACLE_RADIUS_M = (0.2, 0.5)      # metres — random radius per obstacle


def _get_base_map_info(map_name):
    """
    Loads map_name through the gym exactly as training does, and reads the
    metadata (spec.to_dict()) and the ALREADY-LOADED occupancy grid array
    directly off the Track object.
    """
    from f1tenth_gym.envs.env_config import EnvConfig
    tmp_env = gym.make("f1tenth_gym:f1tenth-v0", config=EnvConfig(map_name=map_name, num_agents=1),
                        disable_env_checker=True)  # this gym reuses obs buffers by design — the checker just warns about that, so skip it
    try:
        track = tmp_env.unwrapped.track
        waypoints = np.column_stack([track.centerline.xs, track.centerline.ys])
        meta = track.spec.to_dict()
        occ  = np.array(track.occupancy_map)
        return meta, waypoints, occ
    finally:
        tmp_env.close()


def _world_to_pixel(x, y, resolution, origin, height):
    """
    World (x, y) metres -> pixel (col, row).
    This gym's occupancy_map needs no row flip: row 0 is already the low-y edge.
    """
    # ═════════════════════════════════════════════════════════════════
    # ✏️  TODO 1 — World <-> Pixel Coordinate Conversion
    # ═════════════════════════════════════════════════════════════════

    # Convert a world-frame point (x, y) in metres to a pixel (col, row):
    # shift by the map origin, divide by the resolution, and cast to int.
    # This gym's grid needs no row flip (see Section 2.1).

    # ─────────────────────────────────────────────────────────────────





    # ═════════════════════════════════════════════════════════════════


def _sample_centerline_point(waypoints, track_length_frac_range=(0.15, 0.95)):
    """Pick a random point along the centerline, skipping the region near spawn."""
    # ═════════════════════════════════════════════════════════════════
    # ✏️  TODO 2 — Sample a Random Centerline Point
    # ═════════════════════════════════════════════════════════════════

    # Pick a random waypoint index within track_length_frac_range (expressed
    # as a fraction of len(waypoints)), skipping the region near spawn, and
    # return that waypoint's (x, y).

    # ─────────────────────────────────────────────────────────────────





    # ═════════════════════════════════════════════════════════════════


def generate_obstacle_map(meta, occ_array, waypoints, variant_id):
    """
    Paints 1-3 randomly placed, randomly sized circular obstacles directly
    onto a copy of the occupancy grid (numpy array — works regardless of
    whether it's boolean, 0/1, or 0/255), then saves the result as a real
    PNG + yaml pair in OUTPUT_DIR. Returns a map_name string (full path, no
    extension) for use with F1TenthObstacleWrapper.
    """
    resolution = float(meta["resolution"])
    origin     = meta["origin"]
    height, width = occ_array.shape

    occ = occ_array.copy()
    # Detect the grid's value convention at runtime rather than assuming one:
    # raw grayscale (0-255) uses a LOW value for occupied; a normalized/boolean
    # grid (max <= 1) uses a HIGH value (1 / True) for occupied.
    occupied_value = 0 if occ.max() > 1.5 else occ.dtype.type(1)

    yy, xx = np.ogrid[:height, :width]
    n_obstacles = np.random.randint(MIN_OBSTACLES, MAX_OBSTACLES + 1)
    for _ in range(n_obstacles):
        # ═════════════════════════════════════════════════════════════════
        # ✏️  TODO 3 — Paint One Circular Obstacle onto the Occupancy Grid
        # ═════════════════════════════════════════════════════════════════

        # For this obstacle:
        #  1. Sample a centerline point (TODO 2) -> wx, wy
        #  2. Pick a random radius from OBSTACLE_RADIUS_M -> radius_m
        #  3. Convert to pixels with _world_to_pixel (TODO 1) -> cx, cy;
        #     convert radius_m to pixels -> r_px
        #  4. Build a circular mask of the pixels within r_px of (cx, cy)
        #  5. Mark those pixels as occupied (occupied_value)

        # ─────────────────────────────────────────────────────────────────





        # ═════════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    # Convert to a standard 0-255 grayscale PNG regardless of the grid's
    # original dtype/range.
    occ_float = occ.astype(np.float32)
    if occ_float.max() <= 1.5:
        occ_float = occ_float * 255.0
    occ_uint8 = np.clip(occ_float, 0, 255).astype(np.uint8)

    # Flip before saving: the gym stores occupancy_map flipped and flips the
    # image again when it loads our file.
    img_out = Image.fromarray(np.flipud(occ_uint8), mode="L")

    new_name      = f"{BASE_MAP_NAME}_obs{variant_id}"
    new_img_path  = OUTPUT_DIR / f"{new_name}.png"
    # Track.from_track_path() does: Path(map_name).parent / f"{Path(map_name).stem}_map.yaml"
    # so the yaml MUST be named "<name>_map.yaml", while the returned map_name
    # itself stays WITHOUT that suffix (so .stem correctly equals new_name).
    new_yaml_path = OUTPUT_DIR / f"{new_name}_map.yaml"
    new_cl_path   = OUTPUT_DIR / f"{new_name}_centerline.csv"

    img_out.save(new_img_path)

    # Minimal YAML with no "name" key: the gym sets the map name from the
    # file stem itself.
    new_meta = {
        "image":           new_img_path.name,
        "resolution":      float(meta["resolution"]),
        "origin":          [float(v) for v in meta["origin"]],
        "negate":          int(meta.get("negate", 0)),
        "occupied_thresh": float(meta.get("occupied_thresh", 0.45)),
        "free_thresh":     float(meta.get("free_thresh", 0.196)),
    }
    with open(new_yaml_path, "w") as f:
        yaml.safe_dump(new_meta, f)

    # The gym needs a centerline CSV next to each map (it drives frenet_pose,
    # our progress reward). Same track, so reuse the base centerline (x_m, y_m).
    np.savetxt(new_cl_path, np.asarray(waypoints, dtype=np.float64),
               delimiter=",", header="x_m,y_m", comments="")

    return str(OUTPUT_DIR / new_name)   # NOT new_yaml_path — must match "<name>_map.yaml" pattern


OUTPUT_DIR = Path("/content/obstacle_maps")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Obstacle map generator defined.")


## 2.4 Pre-Generate the Obstacle Map Pool 🏗️

Run this once — it reads the base map's occupancy grid directly from memory, then saves `NUM_VARIANTS` randomized obstacle maps as real files we control.

A sanity check at the end confirms the first variant's files genuinely exist on disk before anything downstream tries to load them.

In [ ]:
# ── Pre-generate the pool of randomized obstacle maps ──────────────────────────
base_meta, waypoints, base_occ = _get_base_map_info(BASE_MAP_NAME)
print(f"Loaded {len(waypoints)} centerline waypoints from {BASE_MAP_NAME}")
print(f"Saving generated variants to: {OUTPUT_DIR}")

OBSTACLE_MAP_POOL = []
for i in range(NUM_VARIANTS):
    name = generate_obstacle_map(base_meta, base_occ, waypoints, variant_id=i)
    OBSTACLE_MAP_POOL.append(name)

print(f"Generated {len(OBSTACLE_MAP_POOL)} obstacle map variants:")
print(f"  {OBSTACLE_MAP_POOL[:2]} ... {OBSTACLE_MAP_POOL[-2:]}")

# Sanity check: the first variant's files exist, named the way the gym expects.
_check_img  = Path(OBSTACLE_MAP_POOL[0] + ".png")
_check_yaml = Path(OBSTACLE_MAP_POOL[0] + "_map.yaml")
print(f"First variant image exists: {_check_img.exists()}  ({_check_img})")
print(f"First variant yaml exists:  {_check_yaml.exists()}  ({_check_yaml})")

_check_cl = Path(OBSTACLE_MAP_POOL[0] + "_centerline.csv")
print(f"First variant centerline exists: {_check_cl.exists()}  ({_check_cl})")

# Load variant 0 the same way the training workers will, so any problem with
# the generated files shows up here.
from f1tenth_gym.envs.track import Track as _Track
_t = _Track.from_track_path(Path(OBSTACLE_MAP_POOL[0]))
_same = int((np.array(_t.occupancy_map) != base_occ).sum())
print(f"Variant 0 loads OK. Pixels differing from the base map (should be ~ the obstacle area, "
      f"NOT most of the map): {_same:,} of {base_occ.size:,}")
assert _same < 0.05 * base_occ.size, "Reloaded map differs a lot from base map — orientation bug"


### Sanity Check — World-to-Pixel Transform ✅

This gym's `occupancy_map` array does **not** need a row-flip (row 0 already corresponds to the low-y edge), and `_world_to_pixel()` reflects this. Run this cell to check that the red dots trace the track exactly before trusting the generated obstacle variants.

In [ ]:
# ── Sanity check: overlay the transformed centerline on the occupancy grid ────
# If the world-to-pixel conversion is correct, the red dots should trace
# exactly along the track shown in gray. Any offset, scale mismatch, or
# flip will show up immediately as red dots drifting away from the track.
height, width = base_occ.shape
resolution = float(base_meta["resolution"])
origin     = base_meta["origin"]

pixel_pts = np.array([_world_to_pixel(x, y, resolution, origin, height) for x, y in waypoints])

plt.figure(figsize=(10, 8))
plt.imshow(base_occ, cmap='gray')
plt.scatter(pixel_pts[:, 0], pixel_pts[:, 1], s=3, c='red', label='centerline (transformed)')
plt.legend()
plt.title("If red dots don't trace the track exactly, the transform has a bug")
plt.show()

print(f"occ shape (height, width): {base_occ.shape}")
print(f"resolution: {resolution}")
print(f"origin: {origin}")
print(f"waypoint x range: {waypoints[:,0].min():.2f} to {waypoints[:,0].max():.2f}")
print(f"waypoint y range: {waypoints[:,1].min():.2f} to {waypoints[:,1].max():.2f}")
print(f"expected width from x-range:  {(waypoints[:,0].max()-waypoints[:,0].min())/resolution:.0f} px")
print(f"expected height from y-range: {(waypoints[:,1].max()-waypoints[:,1].min())/resolution:.0f} px")


## 2.5 Sanity Check — Visualize a Few Variants 👀

Before training on these, look at a few to confirm the obstacles landed on
the track (not off-map or overlapping the spawn region).

In [ ]:
# ── Visualize a few generated variants ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_variants = np.random.choice(OBSTACLE_MAP_POOL, size=4, replace=False)
for ax, name in zip(axes, sample_variants):
    img_path = Path(name + ".png")   # variants are saved as "<map_name>.png"
    img = Image.open(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(Path(name).name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()
print("If the black blobs look like they're sitting on the track (not off in")
print("empty space or right on top of the start line), the generator is working.")


---

# **Section 3 — Reward for Obstacle Avoidance** 🎯

We reuse two reward terms from Learn to Race as-is, and **drop one on purpose**:

- **Keep: progress + speed bonus** — same as always
- **Keep: wall/obstacle proximity penalty** — the obstacle lives in the same
  occupancy grid as the walls, so this term already discourages hitting it,
  with no new code needed
- **Keep (light touch): steering smoothness** — just enough to avoid jittery
  steering, without fighting the sharp swerves genuinely needed to dodge something
- **Drop: centerline lateral-error penalty** — this term punishes leaving the
  racing line, which is exactly what the car must do to avoid an obstacle
  sitting on that line. Keeping it would make the reward fight itself.

---


$$r_t = 10\,\Delta_{\text{progress}} + 0.03\,\tfrac{v}{v_{\max}} - 0.02\,|\Delta\theta| - 0.1\max(0,\ 0.5 - d_{\min}) - 30\cdot\mathbb{1}[\text{collision}]$$

---


---

# **Section 4 — The Obstacle Avoidance Wrapper** 🧩

We will also be using the same observation and action space as the prior modules. The only structural difference:
**every `reset()` builds (or reuses from a cache) an env pointed at a randomly
chosen obstacle-map variant.**

> **📌 Performance note:** each built simulator holds a full copy of the map
> in RAM, so caching every variant in every worker would exhaust memory. Each
> worker keeps only the `MAX_CACHED_ENVS` most recently used simulators (LRU)
> and rebuilds others on demand (~0.3s).


In [ ]:
# ── F1TenthObstacleWrapper ──────────────────────────────────────────────────────
# A fresh, randomly-chosen obstacle map variant is loaded at EVERY reset() —
# the car never sees the same obstacle layout twice, forcing it to react to
# the lidar rather than memorize a fixed detour.

STEER_SMOOTH_COEF   = 0.02      # light touch — don't fight necessary swerves
WALL_PROX_COEF      = 0.1
WALL_PROX_THRESHOLD = 0.5

# Each cached simulator holds a full copy of the map (tens to hundreds of MB),
# so each worker keeps only a few alive (LRU); a cache miss costs ~0.3 s to
# rebuild, negligible next to an episode of hundreds of steps.

MAX_CACHED_ENVS     = 3


import gc
from collections import OrderedDict


class F1TenthObstacleWrapper(gym.Env):
    """
    Observation (110-dim, all in [0,1]) and action space — identical to
    Learn to Drive / Learn to Race. Only the map (via reset) and reward differ.
    """

    def __init__(self, obstacle_map_pool, num_beams=108, render_mode=None):
        super().__init__()
        self.obstacle_map_pool = obstacle_map_pool
        self.render_mode       = render_mode
        self.num_beams         = num_beams
        self.lidar_clip        = 10.0
        self.max_speed         = 6.0
        self.min_speed         = 1.5
        self.max_steps         = 10000
        self.current_step      = 0

        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0, shape=(num_beams + 2,), dtype=np.float32
        )
        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0, shape=(2,), dtype=np.float32
        )

        self.env = None   # points at whichever cached env is active this episode
        self.max_steer  = 0.4189   # gym default; refreshed once self.env exists
        self._env_cache = OrderedDict()   # map_name -> built env, LRU-capped at
                                          # MAX_CACHED_ENVS so memory stays flat

    # ----------------------------------------------------------

    def _build_env(self, map_name):
        from f1tenth_gym.envs.env_config import (
            EnvConfig, SimulationConfig, ObservationConfig, ResetConfig,
        )
        from f1tenth_gym.envs.lidar.config import LiDARConfig
        from f1tenth_gym.envs.observation import ObservationType
        from f1tenth_gym.envs.reset import ResetStrategy
        from f1tenth_gym.envs.integrators import IntegratorType
        from f1tenth_gym.envs.dynamic_models import DynamicModel

        env_config = EnvConfig(
            map_name=map_name,
            num_agents=1,
            simulation_config=SimulationConfig(
                timestep=0.01,
                integrator=IntegratorType.RK4,
                dynamics_model=DynamicModel.ST,
                max_laps=5,
            ),
            observation_config=ObservationConfig(type=ObservationType.DIRECT),
            lidar_config=LiDARConfig(num_beams=self.num_beams, range_max=10.0, noise_std=0.0),
            reset_config=ResetConfig(strategy=ResetStrategy.RL_RANDOM_STATIC),
            render_enabled=(self.render_mode is not None),
        )
        env = gym.make("f1tenth_gym:f1tenth-v0", config=env_config, render_mode=self.render_mode,
                        disable_env_checker=True)  # same reused-buffer warning as above, silenced here too
        self.max_steer = float(env.action_space.high[0][0])
        try:
            self.track_length = env.unwrapped.track.centerline.length
        except Exception:
            self.track_length = None
        return env

    def _get_progress(self, raw_obs):
        try:
            frenet = raw_obs["agent_0"].get("frenet_pose", None)
            if frenet is not None and self.track_length:
                s = float(frenet[0])
                return (s % self.track_length) / self.track_length
        except Exception:
            pass
        return 0.0   # no waypoint fallback needed — frenet is always available here

    def _process_obs(self, raw_obs):
        agent = raw_obs["agent_0"]
        scan  = np.array(agent["scan"], dtype=np.float32) / self.lidar_clip
        state = agent["std_state"]
        speed_norm = np.clip(float(state[3]) / self.max_speed, 0.0, 1.0)
        steer_norm = np.clip(
            (float(state[2]) + self.max_steer) / (2 * self.max_steer), 0.0, 1.0
        )
        return np.concatenate([
            scan.astype(np.float32),
            np.array([speed_norm, steer_norm], dtype=np.float32)
        ])

    def _scale_action(self, action):
        steer = float(action[0]) * self.max_steer
        speed = (float(action[1]) + 1.0) * 0.5 * (self.max_speed - self.min_speed) + self.min_speed
        return np.array([[steer, speed]], dtype=np.float32)

    # ----------------------------------------------------------

    def reset(self, seed=None, options=None):
        # ── Pick a random obstacle layout, building it once then reusing it ────
        map_name = str(np.random.choice(self.obstacle_map_pool))
        if map_name in self._env_cache:
            self._env_cache.move_to_end(map_name)          # mark most-recently used
        else:
            while len(self._env_cache) >= MAX_CACHED_ENVS:  # evict least-recently used
                _, old_env = self._env_cache.popitem(last=False)
                old_env.close()
                del old_env
                gc.collect()
            self._env_cache[map_name] = self._build_env(map_name)
        self.env = self._env_cache[map_name]
        self.max_steer = float(self.env.action_space.high[0][0])
        try:
            self.track_length = self.env.unwrapped.track.centerline.length
        except Exception:
            self.track_length = None

        raw_obs, info      = self.env.reset(seed=seed)
        self.current_step  = 0
        self.prev_progress = self._get_progress(raw_obs)

        state = raw_obs["agent_0"]["std_state"]
        self.prev_physical_steer = float(state[2])

        return self._process_obs(raw_obs), info

    def step(self, action):
        physical_action = self._scale_action(action)
        raw_obs, _, done, truncated, info = self.env.step(physical_action)
        self.current_step += 1

        state     = raw_obs["agent_0"]["std_state"]
        speed     = float(state[3])
        collision = bool(raw_obs["agent_0"].get("collision", False))
        scan_raw  = raw_obs["agent_0"]["scan"]
        min_scan  = float(np.min(scan_raw))

        progress       = self._get_progress(raw_obs)
        delta_progress = progress - self.prev_progress
        if delta_progress < -0.5:
            delta_progress += 1.0

        # ═════════════════════════════════════════════════════════════════
        # ✏️  TODO 4 — Reward Shaping for Obstacle Avoidance
        # ═════════════════════════════════════════════════════════════════

        # Build the reward from Section 3:
        #  1. progress + speed bonus
        #  2. light steering-smoothness penalty
        #     (compute steer_delta, then update self.prev_physical_steer!)
        #  3. wall/obstacle proximity penalty when min_scan < WALL_PROX_THRESHOLD
        #  4. big negative reward + done=True on collision

        # ─────────────────────────────────────────────────────────────────





        # ═════════════════════════════════════════════════════════════════

        if self.current_step >= self.max_steps:
            truncated = True

        info["speed"]       = speed
        info["collision"]   = collision
        info["progress"]    = progress
        info["min_scan"]    = min_scan
        info["steer_delta"] = steer_delta

        # Raw values exposed for the video-recording cell (no effect on reward).
        self.last_xy        = (float(state[0]), float(state[1]))
        self.last_speed     = speed
        self.last_collision = collision
        self.last_min_scan  = min_scan

        self.prev_progress = progress
        return self._process_obs(raw_obs), float(reward), done, truncated, info

    def render(self):
        return self.env.render()

    def close(self):
        for cached_env in self._env_cache.values():
            cached_env.close()


print("F1TenthObstacleWrapper defined — a new random obstacle layout every episode.")


### Diagnostic — Test the Wrapper in a Single Process 🔍

Run this BEFORE the main training cell. `SubprocVecEnv` (used when `NUM_ENVS > 1`) hides real exceptions from crashed worker processes behind a generic `ConnectionResetError` — this reproduces the same construction in the main process so any real error is fully visible.

> ⚠️ Do not skip this cell. If it prints `FAILED`, fix that first — the training cell will fail the same way.

In [ ]:
# ── DIAGNOSTIC: test F1TenthObstacleWrapper directly, single process ─────────

test_env = F1TenthObstacleWrapper(obstacle_map_pool=OBSTACLE_MAP_POOL)

for attempt in range(5):
    print(f"--- reset attempt {attempt+1} ---")
    try:
        obs, info = test_env.reset()
        print(f"  OK — obs shape: {obs.shape}, active map: {list(test_env._env_cache.keys())[-1]}")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        break

test_env.close()


---

# **Section 5 — PPO Algorithm (Unchanged)** 🧠

The network, GAE, and clipped loss are identical to our from-scratch PPO in Learn to
Drive — only the environment changed.

> **📌 Note:** Since this is unchanged from your earlier notebooks, feel free to paste in your own working PPO implementation (network, GAE, clipped loss) from Learn to Drive here instead of rewriting it, if you already have one you trust.


In [ ]:
# ── Imports for from-scratch PPO ──────────────────────────────────────────────
import copy
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

# Vectorized environment utilities — run several simulators in parallel and
# normalize rewards for stable gradients. The PPO algorithm itself is ours,
# built from scratch in the sections below.
from stable_baselines3.common.vec_env import (
    SubprocVecEnv, DummyVecEnv, VecMonitor, VecNormalize
)

print('PyTorch version:', torch.__version__)
print('Modules imported — we build PPO ourselves below.')


In [ ]:
# ── Actor-Critic Network ──────────────────────────────────────────────────────
# Two INDEPENDENT [64,64] MLPs — no shared trunk. Sharing weights between the
# actor and critic would let policy gradients and value gradients interfere
# with the same parameters, hurting both objectives. Orthogonal
# initialization with a small gain on the action head keeps initial actions
# near zero, which is important for early training stability.

def _orthogonal_init(layer, gain):
    nn.init.orthogonal_(layer.weight, gain=gain)
    nn.init.constant_(layer.bias, 0.0)

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        # Actor — fully independent MLP, own weights
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.actor_mean = nn.Linear(hidden, act_dim)

        # Critic — fully independent MLP, own weights (no sharing with the actor)
        self.critic_body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.critic = nn.Linear(hidden, 1)

        # Learnable log standard deviation (state-independent)
        self.log_std = nn.Parameter(torch.zeros(act_dim))

        # Orthogonal init gains: sqrt(2) for hidden layers, 0.01 for the
        # action output (near-zero initial actions), 1.0 for the value output
        for layer in self.actor:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        for layer in self.critic_body:
            if isinstance(layer, nn.Linear): _orthogonal_init(layer, gain=2 ** 0.5)
        _orthogonal_init(self.actor_mean, gain=0.01)
        _orthogonal_init(self.critic, gain=1.0)

    def forward(self, obs):
        # ═══════════════════════════════════════════════════════════════
        # ✏️  TODO 5 — Actor-Critic Forward Pass
        # ═══════════════════════════════════════════════════════════════

        # Pass obs through the actor to get the action mean and through the
        # critic to get the state value, then build a Normal distribution
        # from the mean and the learned log-std.
        # Return: dist, value

        # ───────────────────────────────────────────────────────────────





        # ═══════════════════════════════════════════════════════════════
        pass  # remove once you've implemented the steps above

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        """Sample an action (or take the mean if deterministic)."""
        dist, value = self.forward(obs)
        action = dist.mean if deterministic else dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value


print('ActorCritic network defined (separate actor/critic MLPs, orthogonal init).')


## 5.1 Environment Builders 🏭

In [ ]:
# ── Environment factory ───────────────────────────────────────────────────────

def make_env(rank, seed=0):
    """Returns a function that creates one F1TenthObstacleWrapper instance.
    Every instance shares the same OBSTACLE_MAP_POOL but picks independently."""
    def _init():
        env = F1TenthObstacleWrapper(obstacle_map_pool=OBSTACLE_MAP_POOL)
        env.reset(seed=seed + rank)
        return env
    return _init


def make_training_env(num_envs):
    fns = [make_env(i) for i in range(num_envs)]
    env = DummyVecEnv(fns) if num_envs == 1 else SubprocVecEnv(fns)
    env = VecMonitor(env)
    env = VecNormalize(env, norm_obs=False, norm_reward=True, clip_reward=10.0, gamma=0.99)
    return env


def make_eval_env():
    env = DummyVecEnv([make_env(99)])
    env = VecMonitor(env)
    env = VecNormalize(env, norm_obs=False, norm_reward=True, clip_reward=10.0,
                       gamma=0.99, training=False)
    return env


print("Environment builders defined successfully.")


## 5.2 GAE — Generalized Advantage Estimation 📊

In [ ]:
# ── GAE — Generalized Advantage Estimation ────────────────────────────────────
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """
    rewards, values, dones : tensors of shape (T, N)  [T steps, N envs]
    last_value             : tensor (N,) — V(s_T) for bootstrapping
    Returns advantages and returns, both (T, N).
    """
    T, N = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_adv = torch.zeros(N, device=rewards.device)

    # ═══════════════════════════════════════════════════════════════
    # ✏️  TODO 6 — Compute GAE Advantages
    # ═══════════════════════════════════════════════════════════════

    # Loop backwards from t = T-1 to 0. At each step, take the next value
    # (last_value at the end), discount it only if the episode didn't end,
    # compute the TD residual, then add the decayed advantage from step t+1
    # and store the result in advantages[t].

    # ───────────────────────────────────────────────────────────────





    # ═══════════════════════════════════════════════════════════════

    returns = advantages + values
    return advantages, returns


print('compute_gae defined.')


---

# **Section 6 — Training** 🏋️

## 6.1 Training Configuration ⚙️

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
NUM_ENVS    = 4             # Parallel environments (each keeps a small LRU cache of simulators)
TOTAL_STEPS = 1_000_000     # Total training steps
DEVICE      = "cuda"

print("Training config:")
print(f"  Base map:        {BASE_MAP_NAME}")
print(f"  Obstacle pool:   {len(OBSTACLE_MAP_POOL)} variants")
print(f"  Envs:            {NUM_ENVS}")
print(f"  Steps:           {TOTAL_STEPS:,}")
print(f"  Device:          {DEVICE}")


## 6.2 WandB Login 🔑

In [ ]:
# ── WandB login ───────────────────────────────────────────────────────────────

# Choose option 1 or 2, then paste your WANDB_API key
wandb.login()

## 6.3 Reward Shaping Sweep 🔬 *(Run Before Full Training)*

### Find Your Best Shaping Coefficients with WandB Sweeps

The reward uses three shaping constants: `STEER_SMOOTH_COEF` (steering smoothness), `WALL_PROX_COEF` (wall/obstacle proximity penalty) and `WALL_PROX_THRESHOLD` (the distance at which that penalty switches on).
Rather than guessing all of them, use this sweep to compare combinations with short 50k-step runs.
WandB tries every combination and logs results to a parallel coordinates plot.


<img src="https://raw.githubusercontent.com/dhyeyshah28/RL_for_Roboracer/main/assets/images/hyperparam_sweep.png" width="650">


**How to use:**
1. ✅ Complete **all TODOs** first — the sweep uses your reward, network and GAE implementations
2. Edit the `parameters` lists below to the values you want to compare
3. Run the cell — WandB launches one run per combination
4. Open the sweep link that prints — results update live
5. 🔙 **Go back to Section 4 — The Obstacle Avoidance Wrapper** and put the best values into the three constants at the top of the wrapper cell (`STEER_SMOOTH_COEF`, `WALL_PROX_COEF`, `WALL_PROX_THRESHOLD`), then **re-run that cell**
6. Only then run full training in Section 6.5

> **⚠️ Requires the TODOs complete.** The sweep trains with a short version of the same PPO loop, using your wrapper's `step()`.


In [ ]:
# ── WandB Reward Shaping Sweep ────────────────────────────────────────────────
# Edit the value lists below — every combination will be trained and compared.
# Each run is a short 50k-step version of the same PPO loop used in 6.5.

# ── Define values to sweep (the 3 shaping constants) ─────────────────────────
sweep_config = {
    "method": "grid",
    # Ranked by progress per episode, NOT reward: the swept coefficients are part of
    # the reward itself, so a smaller penalty would score a higher reward without
    # driving any better. Progress (fraction of the lap covered before a crash or
    # the step cap) measures the actual task, and a crash ends the episode early,
    # so collisions are punished automatically.
    "metric": {"name": "eval/mean_progress", "goal": "maximize"},
    "parameters": {
        "steer_smooth_coef":   {"values": [0.02, 0.05]},
        "wall_prox_coef":      {"values": [0.1, 0.2]},
        "wall_prox_threshold": {"values": [0.4, 0.6]},
    }
}

SWEEP_STEPS         = 50_000
SWEEP_ENVS          = 2
SWEEP_N_STEPS       = 1024      # rollout length per env per update
SWEEP_EPOCHS        = 5
SWEEP_MINIBATCH     = 128
SWEEP_EVAL_EPISODES = 3
SWEEP_EVAL_MAX_STEPS = 3000

# ── Training function — uses YOUR wrapper reward ─────────────────────────────
def sweep_train():
    global STEER_SMOOTH_COEF, WALL_PROX_COEF, WALL_PROX_THRESHOLD

    # reinit=True: start a fresh WandB run for every sweep combination.
    run = wandb.init(reinit=True)
    cfg = wandb.config
    print(f"  steer={cfg.steer_smooth_coef} | wall={cfg.wall_prox_coef} | threshold={cfg.wall_prox_threshold}")

    # The wrapper reads these module-level constants inside step(), so write
    # this run's values to them before the envs are built.
    _orig = (STEER_SMOOTH_COEF, WALL_PROX_COEF, WALL_PROX_THRESHOLD)
    STEER_SMOOTH_COEF   = cfg.steer_smooth_coef
    WALL_PROX_COEF      = cfg.wall_prox_coef
    WALL_PROX_THRESHOLD = cfg.wall_prox_threshold

    s_train = s_eval = None
    try:
        torch.manual_seed(42); np.random.seed(42)
        device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')

        # DummyVecEnv (same process) so the constants set above are the ones the envs read
        s_train = DummyVecEnv([make_env(i) for i in range(SWEEP_ENVS)])
        s_train = VecMonitor(s_train)
        s_train = VecNormalize(s_train, norm_obs=False, norm_reward=True,
                               clip_reward=10.0, gamma=0.99)

        s_eval  = DummyVecEnv([make_env(99)])
        s_eval  = VecMonitor(s_eval)
        s_eval  = VecNormalize(s_eval, norm_obs=False, norm_reward=False,
                               clip_reward=10.0, gamma=0.99, training=False)

        obs_dim = s_train.observation_space.shape[0]
        act_dim = s_train.action_space.shape[0]
        net = ActorCritic(obs_dim, act_dim).to(device)
        opt = optim.Adam(net.parameters(), lr=3e-4, eps=1e-5)

        obs = torch.tensor(s_train.reset(), dtype=torch.float32, device=device)
        n_updates = SWEEP_STEPS // (SWEEP_N_STEPS * SWEEP_ENVS)

        for _update in range(n_updates):
            b_obs_l, b_act_l, b_logp_l, b_val_l, b_rew_l, b_done_l = [], [], [], [], [], []
            for _t in range(SWEEP_N_STEPS):
                action, logp, value = net.act(obs)
                next_obs, reward, dones, _ = s_train.step(np.clip(action.cpu().numpy(), -1.0, 1.0))
                b_obs_l.append(obs); b_act_l.append(action)
                b_logp_l.append(logp); b_val_l.append(value)
                b_rew_l.append(torch.tensor(reward, dtype=torch.float32, device=device))
                b_done_l.append(torch.tensor(dones, dtype=torch.float32, device=device))
                obs = torch.tensor(next_obs, dtype=torch.float32, device=device)

            with torch.no_grad():
                _, last_value = net(obs)
            values = torch.stack(b_val_l)
            advantages, returns = compute_gae(torch.stack(b_rew_l), values, torch.stack(b_done_l),
                                              last_value, gamma=0.99, lam=0.95)

            b_obs  = torch.stack(b_obs_l).reshape(-1, obs_dim)
            b_act  = torch.stack(b_act_l).reshape(-1, act_dim)
            b_logp = torch.stack(b_logp_l).reshape(-1)
            b_adv  = advantages.reshape(-1)
            b_ret  = returns.reshape(-1)

            idxs = np.arange(b_obs.shape[0])
            for _epoch in range(SWEEP_EPOCHS):
                np.random.shuffle(idxs)
                for start in range(0, len(idxs), SWEEP_MINIBATCH):
                    mb = idxs[start:start + SWEEP_MINIBATCH]
                    dist, value = net(b_obs[mb])
                    new_logp = dist.log_prob(b_act[mb]).sum(-1)
                    entropy  = dist.entropy().sum(-1).mean()
                    mb_adv   = b_adv[mb]
                    mb_adv   = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)
                    ratio    = torch.exp(new_logp - b_logp[mb])
                    policy_loss = -torch.min(ratio * mb_adv,
                                             torch.clamp(ratio, 0.8, 1.2) * mb_adv).mean()
                    value_loss  = ((value - b_ret[mb]) ** 2).mean()
                    loss = policy_loss + 0.5 * value_loss - 0.001 * entropy
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(net.parameters(), 0.5)
                    opt.step()

        # ── Deterministic evaluation (progress per episode = the sweep metric) ─
        ep_rewards, ep_progress, speeds, collisions = [], [], [], []
        for _ep in range(SWEEP_EVAL_EPISODES):
            eobs = s_eval.reset()
            ep_r, ep_p, prev_p = 0.0, 0.0, None
            for _t in range(SWEEP_EVAL_MAX_STEPS):
                eaction, _, _ = net.act(torch.tensor(eobs, dtype=torch.float32, device=device),
                                        deterministic=True)
                eobs, er, edones, einfos = s_eval.step(np.clip(eaction.cpu().numpy(), -1.0, 1.0))
                ep_r += float(er[0])
                speeds.append(einfos[0].get("speed", 0.0))
                # info["progress"] is the absolute lap position here, so sum the
                # per-step change (with the same wrap-around fix as the reward)
                p = float(einfos[0].get("progress", 0.0))
                if prev_p is not None:
                    d = p - prev_p
                    if d < -0.5:
                        d += 1.0
                    ep_p += d
                prev_p = p
                if edones[0]:
                    collisions.append(float(einfos[0].get("collision", False)))
                    break
            else:
                collisions.append(0.0)
            ep_rewards.append(ep_r)
            ep_progress.append(ep_p)

        wandb.log({
            "eval/mean_progress":  float(np.mean(ep_progress)),
            "eval/mean_reward":    float(np.mean(ep_rewards)),
            "eval/avg_speed":      float(np.mean(speeds)),
            "eval/collision_rate": float(np.mean(collisions)),
        })
    finally:
        if s_train is not None: s_train.close()
        if s_eval  is not None: s_eval.close()
        STEER_SMOOTH_COEF, WALL_PROX_COEF, WALL_PROX_THRESHOLD = _orig   # restore your chosen values
        run.finish()

# ── Launch ───────────────────────────────────────────────────────────────────
n_combos = (len(sweep_config['parameters']['steer_smooth_coef']['values']) *
            len(sweep_config['parameters']['wall_prox_coef']['values']) *
            len(sweep_config['parameters']['wall_prox_threshold']['values']))

sweep_id = wandb.sweep(sweep_config, project="f1tenth-rl-obstacle-sweep")
print(f"Sweep: {n_combos} combinations × {SWEEP_STEPS:,} steps each")
print(f"View at: https://wandb.ai/{wandb.Api().default_entity}/f1tenth-rl-obstacle-sweep/sweeps/{sweep_id}")
wandb.agent(sweep_id, function=sweep_train, count=None)
print()
print("✅ Sweep complete — open the WandB link above to compare results.")
print("   Use the Parallel Coordinates plot to find the best combination.")
print("   Update those values in Section 4 (the wrapper cell), re-run that cell, then run full training in Section 6.5.")


## 6.4 Training Dashboard 📈

In [ ]:
# ── Live Inline Training Plots ────────────────────────────────────────────────
import matplotlib.gridspec as gridspec
from IPython.display import clear_output

_live = {"steps": [], "rew": [], "collision": [], "speed": [], "scan": [], "loss": []}

def draw_live_plots(total_steps):
    if len(_live['steps']) < 2:
        return
    BG = '#0d1117'
    fig = plt.figure(figsize=(14, 6), facecolor=BG)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.35)
    plots = [
        (gs[0,0], _live['rew'],       'ep_reward ↑',        '#00bcd4'),
        (gs[0,1], _live['collision'], 'collision_rate ↓',   '#f44336'),
        (gs[0,2], _live['speed'],     'avg_speed m/s ↑',    '#69ff47'),
        (gs[1,0], _live['scan'],      'avg_min_scan m ↑',   '#ff9800'),
        (gs[1,1], _live['loss'],      'total_loss',         '#e91e63'),
    ]
    for spec, data, title, color in plots:
        ax = fig.add_subplot(spec)
        ax.set_facecolor('#1a1a2e')
        valid = [(s, v) for s, v in zip(_live['steps'], data) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color=color, lw=2, marker='o', markersize=3)
            ax.fill_between(xs, ys, alpha=0.15, color=color)
            lo, hi = min(ys), max(ys); pad = (hi-lo)*0.1 or 0.1
            ax.set_ylim(lo-pad, hi+pad)
        ax.set_title(title, color='white', fontsize=9, pad=4)
        ax.set_xlabel('Steps', color='#888', fontsize=7)
        ax.tick_params(colors='#888', labelsize=7)
        for spine in ax.spines.values(): spine.set_edgecolor('#333')
    step = _live['steps'][-1]
    pct  = step / total_steps * 100
    wandb_url = wandb.run.url if wandb.run else 'not connected'
    fig.suptitle(f'PPO from Scratch — Obstacle Avoidance  |  Step {step:,} / {total_steps:,}  ({pct:.1f}%)'
                 f'   |   WandB → {wandb_url}',
                 color='white', fontsize=9, x=0.02, ha='left')
    clear_output(wait=True)
    plt.show()
    plt.close(fig)

print('Live plot helper defined.')


## 6.5 Train 🚀

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PPO from Scratch — Obstacle Avoidance Training (Run This Single Cell)
# ══════════════════════════════════════════════════════════════════════════════
# Identical from-scratch PPO algorithm to Learn to Drive. Only the
# environment (randomized obstacle maps) and reward (no lateral-error term)
# differ. See those notebooks for full metric-definition notes.
# ──────────────────────────────────────────────────────────────────────────────
from collections import deque

N_STEPS       = 4096
N_EPOCHS      = 10
MINIBATCH     = 256
GAMMA         = 0.99
GAE_LAMBDA    = 0.95
CLIP_EPS      = 0.2
ENT_COEF      = 0.001
VF_COEF       = 0.5
LR            = 3e-4
MAX_GRAD      = 0.5
EVAL_FREQ     = 50_000
EVAL_EPISODES = 5

torch.manual_seed(42)
np.random.seed(42)

run = wandb.init(
    project = 'f1tenth-rl-lecture1',
    name    = f'ppo_scratch_obstacle_{BASE_MAP_NAME}_{TOTAL_STEPS//1000}k',
    config  = dict(n_steps=N_STEPS, n_epochs=N_EPOCHS, gamma=GAMMA,
                   gae_lambda=GAE_LAMBDA, clip_eps=CLIP_EPS, ent_coef=ENT_COEF,
                   lr=LR, base_map=BASE_MAP_NAME, num_envs=NUM_ENVS,
                   num_obstacle_variants=len(OBSTACLE_MAP_POOL)),
)
wandb.define_metric('global_step')
wandb.define_metric('*', step_metric='global_step')
print(f'🔗 WandB → {run.url}\n')

train_env = make_training_env(NUM_ENVS)
eval_env  = make_eval_env()
obs_dim   = train_env.observation_space.shape[0]
act_dim   = train_env.action_space.shape[0]

device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
net    = ActorCritic(obs_dim, act_dim).to(device)
opt    = optim.Adam(net.parameters(), lr=LR, eps=1e-5)
print(net)
print(f'Parameters: {sum(p.numel() for p in net.parameters()):,}\n')

episode_count    = 0
total_collisions = 0

cur_rew  = np.zeros(NUM_ENVS)
cur_spd  = [[] for _ in range(NUM_ENVS)]
cur_scan = [[] for _ in range(NUM_ENVS)]
cur_col  = np.zeros(NUM_ENVS, dtype=bool)
cur_len  = np.zeros(NUM_ENVS, dtype=int)

ep_rew_window = deque(maxlen=100)
ep_len_window = deque(maxlen=100)

_live.update({k: [] for k in _live})
update_speeds = []

obs = train_env.reset()
obs = torch.tensor(obs, dtype=torch.float32, device=device)
global_step    = 0
last_eval_step = 0
n_updates      = TOTAL_STEPS // (N_STEPS * NUM_ENVS)

print(f'Training {TOTAL_STEPS:,} steps = {n_updates} updates — plots update below\n')

for update in range(n_updates):
    mb_obs, mb_act, mb_logp, mb_val, mb_rew, mb_done = [], [], [], [], [], []
    update_speeds = []
    update_scans  = []

    for _ in range(N_STEPS):
        action, logp, value = net.act(obs)
        act_np = action.cpu().numpy()
        clipped_np = np.clip(act_np, -1.0, 1.0)
        next_obs, reward, dones, infos = train_env.step(clipped_np)
        raw_reward = train_env.get_original_reward()

        mb_obs.append(obs)
        mb_act.append(action)
        mb_logp.append(logp)
        mb_val.append(value)
        mb_rew.append(torch.tensor(reward, dtype=torch.float32, device=device))
        mb_done.append(torch.tensor(dones, dtype=torch.float32, device=device))

        cur_rew += raw_reward
        cur_len += 1
        for i, inf in enumerate(infos):
            spd = inf.get('speed', 0.0)
            cur_spd[i].append(spd)
            cur_scan[i].append(inf.get('min_scan', 0.0))
            update_speeds.append(spd)
            update_scans.append(inf.get('min_scan', 0.0))
            if inf.get('collision', False): cur_col[i] = True

        for i, d in enumerate(dones):
            if d and infos[i].get('TimeLimit.truncated', False):
                term_obs = torch.tensor(
                    infos[i]['terminal_observation'], dtype=torch.float32, device=device
                ).unsqueeze(0)
                with torch.no_grad():
                    _, term_value = net(term_obs)
                mb_rew[-1][i] += GAMMA * term_value.item()

        for i, d in enumerate(dones):
            if d:
                episode_count    += 1
                total_collisions += int(cur_col[i])
                ep_rew_window.append(cur_rew[i])
                ep_len_window.append(cur_len[i])

                wandb.log({
                    'racing/avg_speed':       float(np.mean(cur_spd[i]))  if cur_spd[i]  else 0.0,
                    'racing/avg_min_scan':    float(np.mean(cur_scan[i])) if cur_scan[i] else 0.0,
                    'racing/collision_rate':  total_collisions / episode_count,
                    'racing/episode_count':   episode_count,
                    'rollout/ep_rew_mean':    float(np.mean(ep_rew_window)),
                    'rollout/ep_len_mean':    float(np.mean(ep_len_window)),
                    'rollout/ep_len_std':     float(np.std(ep_len_window)) if len(ep_len_window) > 1 else 0.0,
                    'global_step':            global_step,
                })

                cur_rew[i] = 0.0; cur_spd[i] = []; cur_scan[i] = []
                cur_col[i] = False; cur_len[i] = 0

        obs = torch.tensor(next_obs, dtype=torch.float32, device=device)
        global_step += NUM_ENVS

    with torch.no_grad():
        _, last_value = net(obs)
    rewards = torch.stack(mb_rew)
    values  = torch.stack(mb_val)
    dones_t = torch.stack(mb_done)
    advantages, returns = compute_gae(rewards, values, dones_t, last_value,
                                      gamma=GAMMA, lam=GAE_LAMBDA)

    b_obs     = torch.stack(mb_obs).reshape(-1, obs_dim)
    b_act     = torch.stack(mb_act).reshape(-1, act_dim)
    b_logp    = torch.stack(mb_logp).reshape(-1)
    b_adv     = advantages.reshape(-1)
    b_ret     = returns.reshape(-1)
    b_val_old = values.reshape(-1)

    batch_size = b_obs.shape[0]
    idxs = np.arange(batch_size)
    epoch_losses, epoch_pg, epoch_vf = [], [], []
    for epoch in range(N_EPOCHS):
        np.random.shuffle(idxs)
        for start in range(0, batch_size, MINIBATCH):
            mb = idxs[start:start + MINIBATCH]
            dist, value = net(b_obs[mb])
            new_logp = dist.log_prob(b_act[mb]).sum(-1)
            entropy  = dist.entropy().sum(-1).mean()

            mb_adv = b_adv[mb]
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

            ratio       = torch.exp(new_logp - b_logp[mb])
            surr1       = ratio * mb_adv
            surr2       = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * mb_adv
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss  = ((value - b_ret[mb]) ** 2).mean()
            loss        = policy_loss + VF_COEF * value_loss - ENT_COEF * entropy

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD)
            opt.step()

            epoch_losses.append(loss.item())
            epoch_pg.append(policy_loss.item())
            epoch_vf.append(value_loss.item())

    y_true = b_ret.cpu().numpy()
    y_pred = b_val_old.cpu().numpy()
    var_y  = np.var(y_true)
    explained_var = float('nan') if var_y == 0 else 1.0 - np.var(y_true - y_pred) / var_y
    current_std   = net.log_std.exp().mean().item()

    wandb.log({
        'train/loss':                 float(np.mean(epoch_losses)),
        'train/policy_gradient_loss': float(np.mean(epoch_pg)),
        'train/value_loss':           float(np.mean(epoch_vf)),
        'train/std':                  current_std,
        'train/explained_variance':   explained_var,
        'train/learning_rate':        LR,
        'global_step':                global_step,
    })

    if global_step - last_eval_step >= EVAL_FREQ:
        last_eval_step = global_step
        eval_env.ret_rms = copy.deepcopy(train_env.ret_rms)

        eval_rewards, eval_lens = [], []
        for _ in range(EVAL_EPISODES):
            eobs = eval_env.reset()
            edone, erew, elen = False, 0.0, 0
            while not edone:
                eobs_t = torch.tensor(eobs, dtype=torch.float32, device=device)
                with torch.no_grad():
                    eaction, _, _ = net.act(eobs_t, deterministic=True)
                eobs, ereward, edones, _ = eval_env.step(eaction.cpu().numpy())
                erew += float(ereward[0]); elen += 1; edone = bool(edones[0])
            eval_rewards.append(erew); eval_lens.append(elen)
        wandb.log({
            'eval/mean_reward':    float(np.mean(eval_rewards)),
            'eval/mean_ep_length': float(np.mean(eval_lens)),
            'global_step':         global_step,
        })
        print(f'  [eval @ {global_step:,}] reward={np.mean(eval_rewards):.1f} '
              f'ep_len={np.mean(eval_lens):.0f}')

    if episode_count > 0:
        _live['steps'].append(global_step)
        _live['rew'].append(float(np.mean(ep_rew_window)))
        _live['collision'].append(total_collisions / episode_count)
        _live['speed'].append(float(np.mean(update_speeds)) if update_speeds else 0.0)
        _live['scan'].append(float(np.mean(update_scans)) if update_scans else 0.0)
        _live['loss'].append(float(np.mean(epoch_losses)))
        draw_live_plots(TOTAL_STEPS)
        print(f'Update {update+1}/{n_updates} | step {global_step:,} | episodes={episode_count} | '
              f'ep_rew_mean={np.mean(ep_rew_window):.1f} | collision_rate={total_collisions/episode_count:.2f} | '
              f'loss={np.mean(epoch_losses):.4f}')

os.makedirs('models_obstacle', exist_ok=True)
torch.save(net.state_dict(), 'models_obstacle/ppo_scratch_obstacle.pt')
train_env.save('models_obstacle/vecnormalize.pkl')
print('\nSaved → models_obstacle/ppo_scratch_obstacle.pt')
print('Saved → models_obstacle/vecnormalize.pkl')
wandb.finish()
train_env.close()
eval_env.close()


---

# **Section 7 — Save Your Trained Model** 💾

In [ ]:
!zip -r models_obstacle_archive.zip /content/models_obstacle

In [ ]:
from google.colab import files
files.download('models_obstacle_archive.zip')

---

# **Section 8 — Evaluation** 🎬

## 8.1 Evaluate on Fresh, Never-Trained-On Obstacle Layouts 🧪

The real test: generate a handful of **brand-new** obstacle variants that were
**not** in the training pool, and check the policy still avoids them (crash count
and progress per episode below, then a video in 8.2). If it
only works on layouts from the training pool, it memorized rather than learned
to react.

*Note: the eval video keeps recording across several episodes, auto-resetting after each crash/timeout, so a single hard corner doesn't cut the video short.*


In [ ]:
# ── Evaluate on held-out obstacle layouts ──────────────────────────────────────
# Generate a small pool of NEW variants — never seen during training.
# base_meta/waypoints/base_occ come from the pool-generation cell.
HELD_OUT_VARIANTS = 10
held_out_pool = [
    generate_obstacle_map(base_meta, base_occ, waypoints, variant_id=1000 + i)
    for i in range(HELD_OUT_VARIANTS)
]
print(f"Generated {len(held_out_pool)} held-out obstacle layouts (not used in training)")

# ── Run the trained policy on the held-out layouts ────────────────────────────
eval_env_raw = F1TenthObstacleWrapper(obstacle_map_pool=held_out_pool)
eval_net = ActorCritic(obs_dim, act_dim).to('cpu')
eval_net.load_state_dict(torch.load('models_obstacle/ppo_scratch_obstacle.pt', map_location='cpu'))
eval_net.eval()

NUM_EVAL_EPISODES = HELD_OUT_VARIANTS
EVAL_MAX_STEPS    = 5000
all_progress, all_crashed = [], []
for ep in range(NUM_EVAL_EPISODES):
    obs, _ = eval_env_raw.reset()
    done, steps, ep_p, prev_p = False, 0, 0.0, None
    while not done and steps < EVAL_MAX_STEPS:
        with torch.no_grad():
            action, _, _ = eval_net.act(torch.tensor(obs, dtype=torch.float32), deterministic=True)
        obs, _, terminated, truncated, info = eval_env_raw.step(action.numpy())
        done = terminated or truncated; steps += 1
        # info["progress"] is the absolute lap position, so sum per-step changes
        p = float(info.get("progress", 0.0))
        if prev_p is not None:
            d = p - prev_p
            if d < -0.5:
                d += 1.0
            ep_p += d
        prev_p = p
    crashed = bool(info.get("collision", False))
    all_progress.append(ep_p); all_crashed.append(crashed)
    print(f"  Ep {ep+1}: {'CRASH' if crashed else 'NO CRASH'} | progress={ep_p:.3f} | steps={steps}")

print(f"  Average progress:    {np.mean(all_progress):.3f}")
print(f"  Collision-free eps:  {sum(1 for c in all_crashed if not c)}/{NUM_EVAL_EPISODES}")
eval_env_raw.close()


## 8.2 Record Eval Video (Held-Out Layout) 🎥

In [ ]:
# ── Record eval video on a held-out obstacle layout ────────────────────────────
!pip install -q imageio imageio-ffmpeg
import imageio

vid_net = ActorCritic(obs_dim, act_dim).to('cpu')
vid_net.load_state_dict(torch.load('models_obstacle/ppo_scratch_obstacle.pt', map_location='cpu'))
vid_net.eval()

venv = DummyVecEnv([lambda: F1TenthObstacleWrapper(obstacle_map_pool=held_out_pool,
                                                   render_mode='rgb_array')])
venv = VecMonitor(venv)
venv = VecNormalize.load('models_obstacle/vecnormalize.pkl', venv)
venv.training = False; venv.norm_reward = False

frames = []
TOTAL_FRAMES_TARGET = 600
MAX_EPISODES         = 6
STALL_EPS            = 0.02     # metres — treat smaller movement than this as "not moving"
STALL_STEPS_TO_FLAG  = 50

obs = venv.reset()
for episode in range(MAX_EPISODES):
    if len(frames) >= TOTAL_FRAMES_TARGET:
        break
    done = False
    step = 0
    prev_xy = None
    stall_count = 0
    prev_frame = None
    while not done and step < 5000:
        obs_t = torch.tensor(obs, dtype=torch.float32)
        with torch.no_grad():
            action, _, _ = vid_net.act(obs_t, deterministic=True)
        obs, _, dones, infos = venv.step(action.numpy())
        done = dones[0]; step += 1

        raw_env = venv.envs[0]
        xy = getattr(raw_env, "last_xy", None)
        if xy is not None:
            if prev_xy is not None and np.hypot(xy[0] - prev_xy[0], xy[1] - prev_xy[1]) < STALL_EPS:
                stall_count += 1
            else:
                stall_count = 0
            prev_xy = xy

        if step % 25 == 0 or stall_count == STALL_STEPS_TO_FLAG:
            print(f"  step {step:4d}  xy={xy}  speed={getattr(raw_env,'last_speed',None):.2f}  "
                  f"collision={getattr(raw_env,'last_collision',None)}  "
                  f"min_scan={getattr(raw_env,'last_min_scan',None):.2f}  stall_count={stall_count}")
        if stall_count == STALL_STEPS_TO_FLAG:
            print(f"  ⚠️ STALLED SIM — (x, y) hasn't moved more than {STALL_EPS}m in "
                  f"{STALL_STEPS_TO_FLAG} steps. This is a real physics/policy stall, not a video artifact.")

        if step % 5 == 0:
            frame = venv.envs[0].render()
            if frame is not None:
                if prev_frame is not None and np.array_equal(frame, prev_frame):
                    print(f"  ⚠️ STALE RENDER at step {step} — render() returned the exact same "
                          f"pixels as last time. If xy above IS still changing, this is a render bug, "
                          f"not the car actually stopping.")
                prev_frame = frame
                frames.append(frame)
    reason = "collision" if infos[0].get("collision") else (
             "timeout" if step >= 5000 else "lap complete / reset")
    print(f"Episode {episode}: ended after {step} steps ({reason})")
    obs = venv.reset()   # vecenv auto-resets on done too, but this covers the "timeout" case
venv.close()

if frames:
    print(f'✅ Captured {len(frames)} frames')
    imageio.mimsave('eval_render_obstacle.mp4', frames, fps=20)
    from google.colab import files
    files.download('eval_render_obstacle.mp4')
else:
    print('❌ No frames captured')


---

# **Section 9 — Summary** 📝

## What We Built

A policy that reacts to obstacles it has never seen in that exact position
before — not one that memorized a fixed detour:

1. **Randomized obstacle maps** — every episode picks a fresh layout (random
   count, position, and size of obstacles), generated by painting directly
   into the track's occupancy grid.
2. **A reward that doesn't fight the task** — we deliberately dropped the
   centerline-tracking penalty from Learn to Race, since punishing the car for
   leaving the racing line contradicts the exact detour it must make.
3. **Held-out evaluation** — tested on obstacle layouts generated *after*
   training, never part of the training pool, which is the real test of
   whether the policy generalizes.

The PPO algorithm — network, GAE, clipped loss — never changed across any of
the three notebooks in this series. Every meaningful improvement came from the
environment and the reward.

---

# **Coming Up Next** 🔜

This notebook handles obstacles that never move once the episode starts. The
next notebook in this series — **Dynamic Obstacle Avoidance** — keeps the same
PPO algorithm and reward philosophy, but changes what the car has to react to:

| | Static Obstacle Avoidance (this notebook) | Dynamic Obstacle Avoidance (next) |
|---|---|---|
| **Obstacle** | Painted directly into the occupancy grid — fixed for the whole episode | A real second agent (`num_agents=2`) that actively drives around the track |
| **How it's generated** | A pool of pre-baked map variants, picked randomly at `reset()` | A randomized pure-pursuit controller — speed, spawn offset, and lookahead all vary every episode |
| **What the policy must learn** | React once, based on where an obstacle happens to sit | Continuously track and react to something that's also moving — including overtaking |
| **Evaluation** | Held-out map variants never seen in training | No pre-baked pool needed — every `reset()` is already a fresh scenario |

Same lidar-in, action-out interface throughout — only the *world* gets harder.
